# Complete Energy-Profile Comparison — LaTeX tables & boxplot

Same distance-to-real feature scorecard as `results_energy_profile_compare`, but
adds:
1. the **aggregated** scorecard as LaTeX,
2. one scorecard **per process** (separated) as LaTeX, and
3. a **boxplot across all processes** (per method), styled like the sMAE boxplot
   in `results_latex_table`.

Methods: Alpha, Combined-best, Budget (process types, best duration approach per
process), plus Schedule-direct and Profile-generator.

**Metric — paired relative error, not a Wasserstein distance.** For each
`(process, case, sensor)` unit and each per-curve feature `f`:

    err = |f(predicted case) - f(real case)| / mean|f(real)|

i.e. every simulated case is compared against *its own* real counterpart, and the
error is normalised by the typical real magnitude of that sensor so it is
comparable across sensors. Tables report the **median over units**;
**lower = closer to real**. (An earlier version compared the *distribution* of
each feature over cases via a Wasserstein distance — that was unpaired and
weighted by sensor count, and was replaced when the aggregation unit became
`(process, case, sensor)`. The optional `time` column is the one metric that is
still a true W1.)

The first column, **Time**, is the `W1 (time), rel.` metric from
`results_latex_table`: a *paired* per-case comparison of **when** the energy is
drawn, normalized by each case's own real duration. All the other columns are
*distributional* comparisons of a per-curve shape feature (Total, Peak, …).

In [48]:
# ── Config ────────────────────────────────────────────────────────────────────
import os, glob, warnings
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import wasserstein_distance
from IPython.display import display, Markdown
import matplotlib.pyplot as plt, matplotlib.colors as mcolors
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

EXPERIMENT     = 963

# ══ METHODS — turn any row of every table/plot on or off here ═══════════════
# Order here is the order they appear in every table and boxplot.
METHODS = {
    'Baseline':          True,   # per-SENSOR median curve (naive curve generator)
    'Alpha':             True,   # alpha-miner net        + CURVE_APPROACH
    'Combined-best':     True,   # best discovered net    + CURVE_APPROACH
    'Budget':            True,   # best net + duration budgeting + CURVE_APPROACH
    'Schedule-direct':   True,   # real schedule, curves taken directly
    'Profile-generator': True,   # stochastic profile generator
}
# ═══════════════════════════════════════════════════════════════════════════

# ══ FEATURES — turn any column of every table/plot on or off here ══════════
# Order here is the column order in every table.
FEATURE_TOGGLE = {
    'total':     True,    # sum of the curve's samples
    'auc':       False,   # area under the curve, trapezoidal integral of v dt
    'peak':      True,    # max draw
    'mean':      True,
    'median':    False,
    'std':       True,
    'time':      False,   # paired per-case W1 of the value-weighted time distribution
    'ac1':       False,   # lag-1 autocorrelation (smoothness)
    'roughness': False,   # mean |dv| (jaggedness)
    'zero_frac': False,   # share of samples at the idle floor (duty cycle)
}
# ═══════════════════════════════════════════════════════════════════════════

SHOW_OVERALL     = True   # add an 'Overall' column = average across the metric columns
SORT_BY_OVERALL  = True   # order the rows best-to-worst by it

CURVE_APPROACH = 'ml_exog_prev_activity'
# Baseline row: same simulated process as BASELINE_PROCESS_TYPE, but the naive
# per-sensor median curve instead of CURVE_APPROACH. Holding the process model
# fixed means the Baseline vs. that row difference is attributable to the curve
# generator alone. 'baseline' is stored unsuffixed by the pipeline.
BASELINE_CURVE_APPROACH = 'baseline'
BASELINE_PROCESS_TYPE   = 'Budget'

DURATION_MODE  = 'best_by_mae'    # per-process best time prediction, chosen on SELECTION_SPLIT
                                  # (or fix it: 'baseline' / 'ml_global' / 'ml_local')
DURATION_SELECT_METRIC = 'duration_metrics_activity_duration_mae'

# Derived views of METHODS (kept so the rest of the notebook reads unchanged)
PROCESS_TYPES = {k: METHODS.get(k, False) for k in ('Alpha', 'Combined-best', 'Budget')}
EXTRA_METHODS = {k: METHODS.get(k, False) for k in ('Schedule-direct', 'Profile-generator')}
SHOW_BASELINE = METHODS.get('Baseline', False)
DISABLED_MINERS = ['wip_aware', 'wip_branching_aware']
# Model selection ALWAYS happens on TRAIN, independently of SPLIT (which only
# controls what the tables report). Selecting on the split being reported would
# let a model be chosen for fitting the evaluation data.
SELECTION_SPLIT = 'train'

SELECTION_METRIC_BASES = [
    'conformance_metrics_fitness',
    'conformance_metrics_precision',
    'conformance_metrics_generalization',
    'conformance_metrics_simplicity',
]
# The four selection metrics are QUALITY scores (higher = better), so the best
# model is the argmax of their average -- not the argmin used when the criterion
# was expressed as errors.
SELECTION_HIGHER_IS_BETTER = True
COMPOSITE_INCLUDE_TIME = True   # fold W1(time) into the composite boxplot score
SAVE_LATEX = True

results_root = Path('..') / 'results'
runs = sorted([d for d in results_root.iterdir()
               if d.is_dir() and d.name.startswith(f'experiment_{EXPERIMENT}_')])
assert runs, f'No runs for experiment {EXPERIMENT}'
RUN = runs[-1]
print('Using run:', RUN.name)
SCHEDULE_SERIES = {'schedule': 'Schedule-direct', 'stochastic': 'Profile-generator'}
def _suffix_for(approach):
    # The pipeline writes the 'baseline' approach without a suffix.
    return '' if approach in ('baseline', '', None) else f'_{approach}'
_curve_suffix   = _suffix_for(CURVE_APPROACH)
_baseline_suffix = _suffix_for(BASELINE_CURVE_APPROACH)

Using run: experiment_963_20260722_180634


In [49]:
# ── Per-curve features ───────────────────────────────────────────────────────
def curve_features(v, t=None):
    """Per-curve scalars. `t` is the curve's t_minutes axis.

    Two size measures are emitted; FEATURE_TOGGLE picks which is reported.
      'total' -- plain sum of the samples (the reported default).
      'auc'   -- trapezoidal integral of v dt, the true area in kW*min.
    They differ because sum(v) implicitly treats every sample as one minute
    wide and assumes all series share a grid. Measured median dt is 1.000 min
    for `real`, ~1.04 for the petri-net predictions and ~1.21 for the schedule
    / stochastic series, so 'total' carries a ~4% / ~21% grid-dependent offset
    on top of the prediction error, while 'auc' does not. 'auc' falls back to
    the sum when no time axis is available.
    """
    v = np.asarray(v, float)
    if t is not None:
        t = np.asarray(t, float)
        m = np.isfinite(v) & np.isfinite(t)
        v, t = v[m], t[m]
    else:
        v = v[np.isfinite(v)]
    if v.size < 4: return None
    auc = float(np.trapz(v, t)) if (t is not None and t.size == v.size) else float(np.nansum(v))
    rng = np.nanmax(v) - np.nanmin(v)
    zero = np.mean((v - np.nanmin(v)) <= (0.02 * rng if rng > 0 else 1e-9))
    ac1 = pd.Series(v).autocorr(lag=1)
    return {'total': float(np.nansum(v)), 'auc': auc, 'peak': float(np.nanmax(v)),
            'mean': float(np.nanmean(v)), 'median': float(np.nanmedian(v)),
            'std': float(np.nanstd(v)), 'ac1': float(ac1) if np.isfinite(ac1) else np.nan,
            'roughness': float(np.nanmean(np.abs(np.diff(v)))), 'zero_frac': float(zero)}

# 'time' is NOT a per-curve shape feature but a paired per-case metric (see
# w1_time_case below). It is carried in the same grid so it shows up as one more
# column in every table / boxplot; VALUE_FEATURES is what curve_features emits.
_ALL_VALUE_FEATURES = ['total', 'auc', 'peak', 'mean', 'median', 'std', 'ac1', 'roughness', 'zero_frac']
# curve_features() keeps computing all of them (cheap, and keeps the toggle
# reversible); FEATURE_TOGGLE decides what actually reaches the tables.
VALUE_FEATURES = [f for f in _ALL_VALUE_FEATURES if FEATURE_TOGGLE.get(f)]
FEATURES   = (['time'] if FEATURE_TOGGLE.get('time') else []) + VALUE_FEATURES
INCLUDE_TIME = FEATURE_TOGGLE.get('time', False)
FEAT_LABEL = {'time':'Time','total':'Sum','auc':'Area (AUC)','peak':'Max','mean':'Mean','median':'Median',
              'std':'Std','ac1':'AC1','roughness':'Roughness','zero_frac':'Zero-frac'}

def features_long(df_curves, method_label):
    rows = []
    for (sen, cid), g in df_curves.groupby(['sensor', 'case_id']):
        gs = g.sort_values('t_minutes')
        f = curve_features(gs['value'].to_numpy(), gs['t_minutes'].to_numpy())
        if f:
            f.update(sensor=sen, case_id=cid, series=method_label); rows.append(f)
    return rows

# ── W1(time): the timing metric from `results_latex_table` ───────────────────
# For ONE case: compare *when* the energy is drawn -- the value-weighted
# distribution over time -- in real vs. predicted, with time normalized by that
# case's OWN real duration (so a case that simply runs longer isn't penalized
# for its scale, only for shifting its energy around inside the case).
# Degenerate cases are skipped (NaN) instead of exploding the ratio, mirroring
# the `_cv_ok` / degenerate-scale guards used for the value features.
def w1_time_case(t_real, v_real, t_pred, v_pred):
    w_real = np.clip(v_real, 0, None); w_pred = np.clip(v_pred, 0, None)
    if w_real.sum() <= 0 or w_pred.sum() <= 0: return np.nan     # all-zero curve
    dur = float(np.nanmax(t_real))
    if not np.isfinite(dur) or dur <= 1e-6: return np.nan        # zero-length case
    return float(wasserstein_distance(t_real / dur, t_pred / dur,
                                      u_weights=w_real, v_weights=w_pred))

def time_long(df_curves, real_series, pred_series, method_label):
    # One row per (sensor, case) present in BOTH the real and predicted series.
    rows = []
    for sen, sg in df_curves.groupby('sensor'):
        real_by = {c: g.sort_values('t_minutes')
                   for c, g in sg[sg['series'] == real_series].groupby('case_id')}
        pred_by = {c: g.sort_values('t_minutes')
                   for c, g in sg[sg['series'] == pred_series].groupby('case_id')}
        for cid, rg in real_by.items():
            pg = pred_by.get(cid)
            if pg is None or pg.empty: continue
            val = w1_time_case(rg['t_minutes'].to_numpy(float), rg['value'].to_numpy(float),
                               pg['t_minutes'].to_numpy(float), pg['value'].to_numpy(float))
            if np.isfinite(val):
                rows.append({'sensor': sen, 'case_id': cid, 'method': method_label,
                             'w1_time': val})
    return rows

In [50]:
# ── Resolve process-type -> concrete simulation mode per process ─────────────
pe = pd.read_parquet(RUN / 'process_eval_results.parquet')
# NOTE: no display-split prefix is needed here — every column read from `pe` in
# this cell drives a SELECTION, and selections use _sel_prefix (TRAIN).
def parse_mode(m):
    r = str(m)[len('petri_net_'):] if str(m).startswith('petri_net_') else str(m)
    if r.endswith('_ml_plus_global'): return r[:-len('_ml_plus_global')], 'ml_global'
    if r.endswith('_ml_plus_per_act'): return r[:-len('_ml_plus_per_act')], 'ml_local'
    return r, 'baseline'
pe['model'], pe['time_pred'] = zip(*pe['mode'].map(parse_mode))
pe = pe[~pe['model'].isin(DISABLED_MINERS)]
_sel_prefix = SELECTION_SPLIT.lower() + '_'          # selection on TRAIN
_sel_cols = [_sel_prefix + b for b in SELECTION_METRIC_BASES
             if (_sel_prefix + b) in pe.columns]
assert _sel_cols, f'no {_sel_prefix}* selection columns found'
_cands = sorted(set(pe['model'].unique()) - {'alpha', 'budget'} - set(DISABLED_MINERS))
best_miner = {}
for proc, g in pe.groupby('process'):
    gc = g[g['model'].isin(_cands)]
    _sc = gc.groupby('model')[_sel_cols].mean().mean(axis=1) if not gc.empty else None
    best_miner[proc] = (None if _sc is None else
                        (_sc.idxmax() if SELECTION_HIGHER_IS_BETTER else _sc.idxmin()))
_TIME_SUFFIX = {'baseline':'', 'ml_global':'_ml_plus_global', 'ml_local':'_ml_plus_per_act'}
# DURATION_MODE='best_by_mae' picks the time-prediction variant per process by
# DURATION_SELECT_METRIC (activity-duration MAE). That is a SELECTION, so it
# reads the TRAIN column -- the tables below report the chosen variant on test,
# and picking it on test would be selecting on the evaluation split (same rule
# as SELECTION_SPLIT above). MAE rather than WAPE: the comparison is always
# within one process, where the scale is constant, so the normalisation WAPE
# adds buys nothing and MAE stays in interpretable minutes.
_dur_sel_col = _sel_prefix + DURATION_SELECT_METRIC
assert _dur_sel_col in pe.columns, f'{_dur_sel_col} missing'
def model_for(proc, ptype):
    return {'Alpha':'alpha','Budget':'budget'}.get(ptype) or best_miner.get(proc)
def mode_for(proc, ptype):
    model = model_for(proc, ptype)
    if model is None: return None
    if DURATION_MODE in _TIME_SUFFIX: tp = DURATION_MODE
    else:
        sub = pe[(pe['process']==proc)&(pe['model']==model)]
        tp = (sub.loc[sub[_dur_sel_col].idxmin(), 'time_pred']
              if not sub.empty and sub[_dur_sel_col].notna().any() else 'baseline')
    return f'petri_net_{model}{_TIME_SUFFIX[tp]}'

In [51]:
# ── Build unified feature table (real + every method) ────────────────────────
active_ptypes = [t for t in ['Alpha','Combined-best','Budget'] if PROCESS_TYPES.get(t)]
active_extra  = [s for s,lbl in SCHEDULE_SERIES.items() if EXTRA_METHODS.get(lbl)]
METHOD_ORDER  = (['Baseline'] if SHOW_BASELINE else []) + active_ptypes \
                + [SCHEDULE_SERIES[s] for s in active_extra]

all_rows, time_rows = [], []
for proc in sorted(pe['process'].unique()):
    for ptype in active_ptypes:
        mode = mode_for(proc, ptype)
        fp = RUN/'complete_curve_eval_results'/proc/(mode or '')/f'predicted_curves{_curve_suffix}.parquet'
        if not mode or not fp.exists(): continue
        d = pd.read_parquet(fp)
        recs = features_long(d[d['series']=='predicted'], ptype)
        for r in recs: r['process']=proc
        all_rows += recs
        # W1(time): pair each simulated case against its own real counterpart
        if INCLUDE_TIME:
            for r in time_long(d, 'real', 'predicted', ptype): r['process']=proc; time_rows.append(r)
    # Baseline: same simulation mode as BASELINE_PROCESS_TYPE, naive per-sensor
    # median curve. Reuses the mode's own 'real' series for W1(time) pairing.
    if SHOW_BASELINE:
        bmode = mode_for(proc, BASELINE_PROCESS_TYPE)
        bfp = RUN/'complete_curve_eval_results'/proc/(bmode or '')/f'predicted_curves{_baseline_suffix}.parquet'
        if bmode and bfp.exists():
            db = pd.read_parquet(bfp)
            recs = features_long(db[db['series']=='predicted'], 'Baseline')
            for r in recs: r['process']=proc
            all_rows += recs
            if INCLUDE_TIME:
                for r in time_long(db, 'real', 'predicted', 'Baseline'): r['process']=proc; time_rows.append(r)
        else:
            print(f'  ⚠️ Baseline curves missing for {proc} ({bmode}) — row will be empty')

    sfp = RUN/'schedule_profile_eval_results'/proc/'predicted_curves.parquet'
    if not sfp.exists(): continue
    sdf = pd.read_parquet(sfp)
    for r in (rr for rr in features_long(sdf[sdf['series']=='real'], 'real')): r['process']=proc; all_rows.append(r)
    for s in active_extra:
        recs = features_long(sdf[sdf['series']==s], SCHEDULE_SERIES[s])
        for r in recs: r['process']=proc
        all_rows += recs
        if INCLUDE_TIME:
            for r in time_long(sdf, 'real', s, SCHEDULE_SERIES[s]): r['process']=proc; time_rows.append(r)
feat = pd.DataFrame(all_rows)
tim  = pd.DataFrame(time_rows)

# restrict to sensors common to real + every method within each process
keep, keep_t = [], []
for proc, g in feat.groupby('process'):
    sets = [set(g[g.series==m]['sensor'].unique()) for m in (['real']+METHOD_ORDER) if m in set(g.series)]
    common = set.intersection(*sets) if sets else set()
    keep.append(g[g['sensor'].isin(common)])
    if not tim.empty:
        gt = tim[tim['process']==proc]
        keep_t.append(gt[gt['sensor'].isin(common)])
feat = pd.concat(keep, ignore_index=True)
tim  = pd.concat(keep_t, ignore_index=True) if keep_t else tim
print('feat rows:', len(feat), '| W1(time) case rows:', len(tim), '| methods:', METHOD_ORDER)

feat rows: 15254 | W1(time) case rows: 0 | methods: ['Baseline', 'Alpha', 'Combined-best', 'Budget', 'Schedule-direct', 'Profile-generator']


In [52]:
# ── One row per (process, case, sensor, method, feature) — the raw grid ──────
# AGGREGATION UNIT = (process, case, sensor). Previously the unit was the
# (process, sensor) cell: within a cell the feature's distribution over cases
# for `real` was compared to the one for the method via an *unpaired*
# Wasserstein distance, and the cases were consumed there. Two consequences
# made that a poor choice:
#   * a method could match the marginal distribution while being wrong on every
#     individual case (the distributions overlap, the pairs don't), and
#   * the median was over ~71 cells, weighted by SENSOR COUNT, so process_5
#     (27 sensors, 16 cases) carried 38% of every number while process_1
#     (90 cases) carried 1.4%.
# Now each case is compared to its OWN real counterpart and kept as its own
# row, so every (process, case, sensor) is one unit of the median — and the
# per-unit values survive, which is what a bootstrap CI needs.
#
# Value features: paired relative error |f(pred) - f(real)| / mean|f(real)|,
# normalised per sensor by the same scale as before so magnitudes stay
# comparable across sensors and the degenerate-scale guard still applies.
# 'time' is already a paired per-case quantity and enters unchanged.
records = []
for (proc, sen), g in feat.groupby(['process', 'sensor']):
    real_g = g[g.series == 'real'].drop_duplicates('case_id').set_index('case_id')
    for feature in VALUE_FEATURES:
        r = real_g[feature].dropna()
        if len(r) < 2:
            continue
        scale = np.nanmean(np.abs(r.to_numpy())) + 1e-9
        # Degenerate-scale guard: for zero-inflated energy sensors the per-case
        # 'median' (and occasionally others) is ~0 for every real case, so
        # mean|real| collapses and the relative error explodes. Skip those cells
        # rather than emit meaningless ~1e9 values.
        if scale < 1e-6:
            continue
        for m in METHOD_ORDER:
            q = g[g.series == m].drop_duplicates('case_id').set_index('case_id')[feature].dropna()
            shared = r.index.intersection(q.index)
            if len(shared) == 0:
                continue
            err = (q.loc[shared] - r.loc[shared]).abs() / scale
            for cid, v in err.items():
                records.append({'process': proc, 'sensor': sen, 'case_id': cid,
                                'method': m, 'feature': feature, 'rel_err': float(v)})
units = pd.DataFrame(records)

# W1(time): already one paired value per (process, sensor, case, method)
if INCLUDE_TIME and not tim.empty:
    t_units = tim.rename(columns={'w1_time': 'rel_err'}).copy()
    t_units['feature'] = 'time'
    units = pd.concat([units, t_units[['process', 'sensor', 'case_id',
                                       'method', 'feature', 'rel_err']]],
                      ignore_index=True)

print(f'units: {len(units):,} rows | '
      f'{units[["process","case_id","sensor"]].drop_duplicates().shape[0]:,} '
      f'distinct (process, case, sensor)')
print(units.groupby('method').size().reindex(METHOD_ORDER).to_string())


def scorecard(df_units):
    t = (df_units.groupby(['method', 'feature'])['rel_err'].median().unstack('feature')
             .reindex(index=METHOD_ORDER, columns=FEATURES))
    t.columns = [FEAT_LABEL[c] for c in t.columns]
    if SHOW_OVERALL:
        t['Overall'] = t.mean(axis=1)      # simple average across the metric columns
        if SORT_BY_OVERALL:
            t = t.sort_values('Overall')   # best first
    t.index.name = 'Method'
    return t


def style_score(tbl):
    return (tbl.style.format('{:.3f}', na_rep='—')
              .highlight_min(axis=0, props='font-weight:700;background-color:#d6ecff;')
              .set_caption('Median per-case error to real over (process, case, sensor) '
                           '— lower = closer to real'))

units: 51,933 rows | 2,206 distinct (process, case, sensor)
method
Baseline             8779
Alpha                8042
Combined-best        8775
Budget               8779
Schedule-direct      8779
Profile-generator    8779


### What each metric means

Every cell of the grid is one **(process, sensor, method)** pair; the tables show the
**median** of those cells. Two different kinds of quantity share the grid:

**All other columns — paired, per case.** For each case a single scalar is computed
from its curve, for both the real and the predicted version. The number is
`|f(pred) - f(real)| / mean|f(real)|` — the error on that case, expressed as a fraction
of the typical real value for that sensor, so it is comparable across sensors with
wildly different units. Sensors where `mean|f(real)|` collapses to ~0 (zero-inflated)
are dropped rather than reported as huge nonsense.

| Column | Per-curve scalar | Reads as |
|---|---|---|
| `Total` | `sum(v)` | total energy drawn over the case — gets the **overall consumption** right |
| `Peak` | `max(v)` | highest instantaneous load — matters for **peak demand / connection sizing** |
| `Mean` | `mean(v)` | average load level over the case |
| `Median` | `median(v)` | *typical* load level, robust to spikes; for on/off sensors this is essentially the **idle / base level** |
| `Std` | `std(v)` | how much the load swings within a case — **amplitude of the dynamics** |
| `AC1` | `corr(v_t, v_{t-1})` | lag-1 autocorrelation: **persistence/smoothness**. ~1 = smooth ramps, ~0 = noisy sample-to-sample. Low error here = the method reproduces how *gradually* load changes |
| `Roughness` | `mean|Δv|` | mean absolute step between consecutive samples — **jaggedness / switching intensity**. A method that adds high-frequency noise blows up here even if Mean/Total are perfect |
| `Zero-frac` | share of samples within 2 % of the curve's own range above its minimum | **duty cycle**: fraction of the case spent at the floor/idle level vs. actually running |

`AC1` and `Roughness` are the two *shape* metrics (are the dynamics realistic?);
`Total`/`Peak`/`Mean`/`Median`/`Std` are *level* metrics (is the magnitude realistic?);
`Zero-frac` is a *duty-cycle* metric; `Time` is the *timing* metric. Lower is better
everywhere.

## 1 · Aggregated scorecard (all processes)

In [53]:
score_all = scorecard(units)
display(style_score(score_all))

,Sum,Max,Mean,Std,Overall
Method,,,,,
Budget,0.253,0.094,0.106,0.511,0.241
Schedule-direct,0.379,0.089,0.082,0.622,0.293
Baseline,0.281,0.123,0.080,0.703,0.297
Alpha,0.476,0.093,0.104,0.526,0.300
Combined-best,0.474,0.103,0.104,0.533,0.304
Profile-generator,0.379,0.189,0.088,1.600,0.564


## 2 · Per process (separated)

In [54]:
per_process_scores = {}
for proc in sorted(units['process'].unique()):
    s = scorecard(units[units['process']==proc])
    per_process_scores[proc] = s
    display(Markdown(f'### {proc}'))
    display(style_score(s))

### process_1

,Sum,Max,Mean,Std,Overall
Method,,,,,
Profile-generator,0.360,0.255,0.262,0.202,0.270
Combined-best,0.225,0.151,0.537,0.167,0.270
Budget,0.218,0.147,0.611,0.175,0.288
Alpha,0.529,0.195,0.772,0.212,0.427
Schedule-direct,0.759,0.142,0.816,0.691,0.602
Baseline,0.875,0.990,0.918,1.010,0.948


### process_2

,Sum,Max,Mean,Std,Overall
Method,,,,,
Budget,0.363,0.172,0.507,0.256,0.325
Combined-best,0.448,0.203,0.423,0.252,0.332
Alpha,0.465,0.205,0.448,0.287,0.351
Profile-generator,0.695,0.236,0.738,0.220,0.472
Schedule-direct,0.486,0.196,0.769,0.624,0.519
Baseline,1.282,0.539,1.150,0.470,0.861


### process_3

...

,Sum,Max,Mean,Std,Overall
Method,,,,,
Budget,0.250,0.049,0.076,0.595,0.243
Baseline,0.206,0.083,0.035,0.834,0.289
Combined-best,0.414,0.066,0.079,0.630,0.297
Schedule-direct,0.307,0.067,0.053,0.771,0.299
Alpha,0.577,0.050,0.074,0.611,0.328
Profile-generator,0.287,0.188,0.040,1.629,0.536


### process_4_2

,Sum,Max,Mean,Std,Overall
Method,,,,,
Schedule-direct,0.390,0.087,0.075,0.470,0.256
Budget,0.338,0.121,0.097,0.480,0.259
Alpha,0.320,0.133,0.098,0.501,0.263
Combined-best,0.508,0.126,0.097,0.509,0.310
Baseline,0.395,0.156,0.122,0.601,0.319
Profile-generator,0.440,0.121,0.108,1.564,0.558


### process_5

,Sum,Max,Mean,Std,Overall
Method,,,,,
Baseline,0.122,0.103,0.053,0.745,0.256
Budget,0.144,0.102,0.090,0.815,0.288
Schedule-direct,0.384,0.097,0.073,0.622,0.294
Alpha,0.529,0.096,0.091,0.662,0.344
Combined-best,0.787,0.105,0.093,0.756,0.435
Profile-generator,0.390,0.420,0.069,2.654,0.883


## 3 · LaTeX — aggregated + one table per process

In [55]:
# ── LaTeX: table* + minipage, caption on top, note at the bottom ─────────────
# Layout knobs — change these if the table doesn't sit right on the page.
LATEX_MINIPAGE   = '13cm'    # width of the centring minipage
LATEX_NOTE_WIDTH = '13cm'    # width of the footnote parbox below the table
LATEX_TYPE_COL   = '2.0cm'   # 'Type' column width
LATEX_METHOD_COL = '3.4cm'   # 'Method' column width (wraps to 2 lines on its own)
GROUP_BY_TYPE    = True      # group rows under Type; False = flat sort by Overall

# Display names and the Type each method belongs to.
METHOD_TEX = {
    'Baseline':          'Baseline',
    'Alpha':             'Alpha Petri Net',
    'Combined-best':     'Best Petri Net',
    'Budget':            'Best Petri Net + Budget',
    'Schedule-direct':   'Schedule-direct',
    'Profile-generator': 'Profile-generator',
}
METHOD_TYPE = {
    'Baseline':          'Baseline',
    'Alpha':             'Process model',
    'Combined-best':     'Process model',
    'Budget':            'Process model',
    'Schedule-direct':   'Schedule-based',
    'Profile-generator': 'Schedule-based',
}
TYPE_ORDER = ['Baseline', 'Process model', 'Schedule-based']

METHOD_NOTE = (
    r'\textbf{Baseline}: the median curve of each sensor, reused for every case. '
    r'\textbf{Alpha Petri Net}: net discovered by the alpha miner. '
    r'\textbf{Best Petri Net}: best discovered net per process, selected on the '
    r'training split by the mean of Fitness, Precision, Generalization and Simplicity. '
    r'\textbf{Best Petri Net + Budget}: the same net, with each case generated to match '
    r'its predicted total-duration budget. '
    r'\textbf{Schedule-direct}: curves placed directly on the real schedule. '
    r'\textbf{Profile-generator}: stochastic profile generator. '
    r'The three Petri-net rows use the ' + CURVE_APPROACH.replace('_', r'\_') +
    r' curve predictor.'
)


def _row_order(tbl):
    """Rows grouped by Type (best Overall first inside each group), or flat."""
    if not GROUP_BY_TYPE:
        return list(tbl.index)
    key = 'Overall' if 'Overall' in tbl.columns else tbl.columns[0]
    order = []
    for typ in TYPE_ORDER:
        grp = [m for m in tbl.index if METHOD_TYPE.get(m) == typ]
        order += sorted(grp, key=lambda m: (pd.isna(tbl.loc[m, key]), tbl.loc[m, key]))
    return order + [m for m in tbl.index if m not in order]


def to_latex_score(tbl, caption, label, note_extra=''):
    metric_cols = list(tbl.columns)
    best = {c: tbl[c].dropna().min() for c in metric_cols if tbl[c].notna().any()}

    def cell(m, c):
        v = tbl.loc[m, c]
        if pd.isna(v):
            return '--'
        s = f'{v:.3f}'
        return r'\textbf{' + s + '}' if abs(v - best.get(c, np.inf)) < 1e-9 else s

    order = _row_order(tbl)
    colfmt = (f'p{{{LATEX_TYPE_COL}}}|p{{{LATEX_METHOD_COL}}}|'
              + '|'.join(['c'] * len(metric_cols)))
    head = (r'\textbf{Type} & \textbf{Method} & '
            + ' & '.join(r'\textbf{' + str(c) + '}' for c in metric_cols) + r' \\')

    body = []
    i = 0
    while i < len(order):
        m = order[i]
        typ = METHOD_TYPE.get(m, '')
        span = 1
        if GROUP_BY_TYPE:
            while i + span < len(order) and METHOD_TYPE.get(order[i + span]) == typ:
                span += 1
        for k in range(span):
            mm = order[i + k]
            tcell = (r'\multirow{' + str(span) + r'}{*}{' + typ + '}') if (k == 0 and GROUP_BY_TYPE) \
                    else ('' if GROUP_BY_TYPE else typ)
            body.append(f'{tcell} & {METHOD_TEX.get(mm, mm)} & '
                        + ' & '.join(cell(mm, c) for c in metric_cols) + r' \\')
        if GROUP_BY_TYPE and i + span < len(order):
            body.append(r'\midrule')
        i += span

    return '\n'.join([
        r'\begin{table*}[H]', r'\centering', '',
        f'\\begin{{minipage}}{{{LATEX_MINIPAGE}}}', r'\centering', '',
        r'\captionsetup{', r'    justification=centering,',
        r'    singlelinecheck=false,', r'    format=plain', r'}', '',
        r'\caption{' + caption + '}', r'\label{' + label + '}', '',
        r'\vspace{-0.5em}', '',
        f'\\begin{{tabular}}{{{colfmt}}}', r'\toprule', head, r'\midrule',
        *body, r'\bottomrule', r'\end{tabular}', '',
        r'\vspace{0.5em}', '',
        f'\\parbox{{{LATEX_NOTE_WIDTH}}}{{%', r'\footnotesize',
        METHOD_NOTE + (' ' + note_extra if note_extra else ''),
        '}', '', r'\end{minipage}', '', r'\end{table*}',
    ])


_metric_note = (r'Cells are the median over (process, case, sensor) of the paired per-case '
                r'relative error $|f(\mathrm{pred})-f(\mathrm{real})|/\overline{|f(\mathrm{real})|}$. '
                r'Lower is better; \textbf{bold} = best per column. '
                r'Overall is the average across the metric columns.')

tex_all = to_latex_score(
    score_all,
    caption='Complete energy-profile comparison, all processes.',
    label=f'tab:energy_profile_all_{EXPERIMENT}',
    note_extra=_metric_note)
print('% ===== ALL PROCESSES =====')
print(tex_all)

for proc, s in per_process_scores.items():
    print(f'\n% ===== {proc} =====')
    print(to_latex_score(
        s,
        caption=f'Complete energy-profile comparison for {proc}.'.replace('_', r'\_'),
        label=f'tab:energy_profile_{proc}_{EXPERIMENT}',
        note_extra=_metric_note))

% ===== ALL PROCESSES =====
\begin{table*}[H]
\centering

\begin{minipage}{13cm}
\centering

\captionsetup{
    justification=centering,
    singlelinecheck=false,
    format=plain
}

\caption{Complete energy-profile comparison, all processes.}
\label{tab:energy_profile_all_963}

\vspace{-0.5em}

\begin{tabular}{p{2.0cm}|p{3.4cm}|c|c|c|c|c}
\toprule
\textbf{Type} & \textbf{Method} & \textbf{Sum} & \textbf{Max} & \textbf{Mean} & \textbf{Std} & \textbf{Overall} \\
\midrule
\multirow{1}{*}{Baseline} & Baseline & 0.281 & 0.123 & \textbf{0.080} & 0.703 & 0.297 \\
\midrule
\multirow{3}{*}{Process model} & Best Petri Net + Budget & \textbf{0.253} & 0.094 & 0.106 & \textbf{0.511} & \textbf{0.241} \\
 & Alpha Petri Net & 0.476 & 0.093 & 0.104 & 0.526 & 0.300 \\
 & Best Petri Net & 0.474 & 0.103 & 0.104 & 0.533 & 0.304 \\
\midrule
\multirow{2}{*}{Schedule-based} & Schedule-direct & 0.379 & \textbf{0.089} & 0.082 & 0.622 & 0.293 \\
 & Profile-generator & 0.379 & 0.189 & 0.088 & 1.600 & 0.564 \\
\